# Phase 1: Data Understanding — Telco Customer Churn

## 1. Objective
The objective of **Phase 1** is to systematically examine the raw Telco Customer Churn dataset, understand its structure and feature types, detect anomalies (including subtle "hidden" nulls), analyze the target distribution (`Churn`), and define actionable requirements for data cleaning and feature engineering.

### Workflow Checklist:
- [x] Load raw dataset (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)
- [x] Inspect dimensions, feature names, and inferred data types
- [x] Classify columns by domain semantics (Demographics, Services, Account, Target)
- [x] Identify explicit and implicit missing values (e.g., whitespace in `TotalCharges`)
- [x] Analyze target variable balance (`Churn` ratio)
- [x] Profile unique values and feature distributions
- [x] Formulate recommendations for Phase 2 (Data Cleaning)

In [ ]:
# 2. Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [2]:
# 3. Load Raw Data
raw_data_path = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(raw_data_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()

Dataset Shape: 7043 rows, 21 columns



## 4. Dataset Schema & Column Taxonomy
The dataset contains 21 columns representing customer demographics, account settings, subscribed services, and churn status.

In [ ]:
# Inspect data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Check customerID uniqueness
n_unique_ids = df['customerID'].nunique()
n_duplicates = df['customerID'].duplicated().sum()
print(f"Total records: {len(df)}")
print(f"Unique customer IDs: {n_unique_ids}")
print(f"Duplicate customer IDs: {n_duplicates}")

Total records: 7043
Unique customer IDs: 7043
Duplicate customer IDs: 0


### Feature Grouping by Domain:
1. **Identifier**: `customerID` (Unique per customer, drop for ML)
2. **Demographics**: `gender`, `SeniorCitizen`, `Partner`, `Dependents`
3. **Phone Services**: `PhoneService`, `MultipleLines`
4. **Internet & Add-ons**: `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
5. **Account & Billing**: `Contract`, `PaperlessBilling`, `PaymentMethod`, `tenure`, `MonthlyCharges`, `TotalCharges`
6. **Target**: `Churn` (Binary: 'Yes' / 'No')

## 5. Missing Values & Hidden Anomaly Detection
Standard `.isnull().sum()` check often reports 0 missing values for this dataset because whitespace characters `' '` are stored as strings in `TotalCharges`.

In [5]:
# Check explicit nulls
print("Explicit null count per column:")
print(df.isnull().sum())

# Check implicit whitespace nulls in object columns
whitespace_cols = {}
for col in df.select_dtypes(include=['object']).columns:
    empty_count = (df[col].astype(str).str.strip() == '').sum()
    if empty_count > 0:
        whitespace_cols[col] = empty_count

print("\nImplicit whitespace empty strings detected:")
for col, count in whitespace_cols.items():
    print(f"  - {col}: {count} empty string rows")

Explicit null count per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Implicit whitespace empty strings detected:
  - TotalCharges: 11 empty string rows


In [6]:
# Inspect the 11 records with empty TotalCharges
empty_tc_rows = df[df['TotalCharges'].astype(str).str.strip() == '']
empty_tc_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'PaymentMethod', 'Churn']]

> **Key Discovery**: All 11 records with blank `TotalCharges` have `tenure == 0`! These are newly registered customers who have not yet completed their first billing cycle. In Phase 2, these can be safely imputed with `0.0` or their `MonthlyCharges`.

## 6. Target Variable Analysis (`Churn`)
Let's assess the distribution of the target variable to determine the extent of class imbalance.

In [7]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage (%)': churn_pct.round(2)
})
print("Churn Distribution:")
print(churn_summary)

Churn Distribution:
       Count  Percentage (%)
Churn                       
No      5174           73.46
Yes     1869           26.54


- **Non-churners (`No`)**: 5,174 (73.46%)
- **Churners (`Yes`)**: 1,869 (26.54%)
- **Class Ratio**: ~2.77 : 1 (Moderate class imbalance)
- **Implication**: Standard accuracy alone will be misleading; precision, recall, PR-AUC, ROC-AUC, and threshold tuning (or resampling/class weighting) will be essential.

## 7. Numerical & Categorical Features Profiling

In [8]:
# Summary statistics for numerical variables
df[['tenure', 'MonthlyCharges']].describe().T

In [9]:
# Unique values across all categorical features
cat_cols = df.select_dtypes(include=['object']).columns.drop(['customerID', 'TotalCharges'])
print("Categorical Features and Unique Cardinalities:\n")
for col in cat_cols:
    print(f"{col} ({df[col].nunique()} unique): {df[col].unique().tolist()}")

Categorical Features and Unique Cardinalities:

gender (2 unique): ['Female', 'Male']
Partner (2 unique): ['Yes', 'No']
Dependents (2 unique): ['No', 'Yes']
PhoneService (2 unique): ['No', 'Yes']
MultipleLines (3 unique): ['No phone service', 'No', 'Yes']
InternetService (3 unique): ['DSL', 'Fiber optic', 'No']
OnlineSecurity (3 unique): ['No', 'Yes', 'No internet service']
OnlineBackup (3 unique): ['Yes', 'No', 'No internet service']
DeviceProtection (3 unique): ['No', 'Yes', 'No internet service']
TechSupport (3 unique): ['No', 'Yes', 'No internet service']
StreamingTV (3 unique): ['No', 'Yes', 'No internet service']
StreamingMovies (3 unique): ['No', 'Yes', 'No internet service']
Contract (3 unique): ['Month-to-month', 'One year', 'Two year']
PaperlessBilling (2 unique): ['Yes', 'No']
PaymentMethod (4 unique): ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Churn (2 unique): ['No', 'Yes']


## 8. Summary of Findings & Next Steps for Phase 2

| Observation | Finding | Action in Phase 2 (Data Cleaning) |
| :--- | :--- | :--- |
| **Dataset Dimensions** | 7,043 rows, 21 columns | Dataset is clean in shape, no duplicate rows. |
| **Identifiers** | `customerID` has 7,043 unique values | Remove from feature set as it has no predictive signal. |
| **TotalCharges Type** | Stored as `object` (string) | Convert to numeric (`float64`). |
| **Hidden Missing Values**| 11 records have blank space `' '` in `TotalCharges` (`tenure == 0`) | Impute `TotalCharges = 0.0` or `TotalCharges = MonthlyCharges`. |
| **SeniorCitizen** | Stored as `int64` (0 or 1) | Treat as binary categorical feature. |
| **Target Variable** | `Churn` is binary ('No': 73.5%, 'Yes': 26.5%) | Encode as binary (0/1); prepare for imbalance handling. |
| **Redundant Categories** | 'No internet service' / 'No phone service' in add-ons | Consolidate or encode cleanly during feature preprocessing. |